# consumer_orders.ipynb

Real-Time Food Delivery Analytics — **ORDERS feed**

Consumes `group_10_orders` from Azure Event Hubs, decodes Avro, flattens, and runs
three tiers of analytics:

| Tier | Query | Window |
|---|---|---|
| Basic        | Orders per zone — tumbling 5-min | 5-min tumbling |
| Intermediate | Cancellation rate per zone — hopping 10-min / 5-min | — |
| Advanced     | Delivery time anomaly detection — 15-min windows | — |

## Prerequisites
1. `producer_orders.ipynb` must have been run (at minimum the Test cell — more
   data = more interesting aggregates).
2. Colab Secret `CONSUMER_CONN_STR` must be set to the Consumer policy
   connection string for namespace `iesstsabbadbaa-grp-06-10`.

## How to run
Run cells top-to-bottom. Sections 1-7 are the standard scaffold (connection,
Spark setup, Avro decode, smoke test). Section 8 contains the analytics.
Each query in Section 8 has a **start cell** followed by a **peek cell** that
waits a few seconds and displays results. Run them in sequence.


## 1. Configuration

In [1]:
from google.colab import userdata

# Azure Event Hub configuration
event_hub_namespace = 'iesstsabbadbaa-grp-06-10'
eventhub_name       = 'group_10_orders'

# Only the Consumer connection string is needed in this notebook.
consumer_eventhub_connection_str = userdata.get('CONSUMER_CONN_STR')

print(f"Consumer notebook configured for orders feed: {eventhub_name}")


Consumer notebook configured for orders feed: group_10_orders


In [2]:
!pip install -q fastavro confluent-kafka


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 71.9 MB/s eta 0:00:00


## 2. Spark Setup

In [3]:
import os
import subprocess

# Fetch the latest Spark 4.x.x version
spark_version = subprocess.run(
    "curl -s https://downloads.apache.org/spark/ | grep -o 'spark-4\\.1\\+\\.[0-9]\\+' | sort -V | tail -1",
    shell=True, capture_output=True, text=True
).stdout.strip()

spark_release = spark_version
hadoop_version = 'hadoop3'

os.environ['SPARK_RELEASE'] = spark_release
os.environ['HADOOP_VERSION'] = hadoop_version
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/{spark_release}-bin-{hadoop_version}"

print(f"Spark release: {spark_release}")


Spark release: spark-4.1.1


In [4]:
# Install Java 21 and Spark 4.x.y
!apt-get update -qq
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/${SPARK_RELEASE}/${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz
!tar xf ${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz

!pip install -q findspark

import findspark
findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
PySpark version: 4.1.1


In [5]:
!echo "==== Java JDK: $JAVA_HOME ===="
!/usr/lib/jvm/java-21-openjdk-amd64/bin/java --version
!echo "==== Spark binaries ===="
!ls ${SPARK_RELEASE}-bin-${HADOOP_VERSION}/bin/ | head -5


==== Java JDK: /usr/lib/jvm/java-21-openjdk-amd64 ====
[0.059s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.059s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk 21.0.10 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)
==== Spark binaries ====
beeline
beeline.cmd
docker-image-tool.sh
find-spark-home
find-spark-home.cmd


## 3. Create Spark Session

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.avro.functions import from_avro

spark = SparkSession \
    .builder \
    .appName("StreamingAVROFromEventHub") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,'
            'org.apache.spark:spark-avro_2.13:4.1.1') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready.")


Spark session ready.


## 4. Avro Schema

In [7]:
# Avro schema as a Python string — MUST exactly match the producer's schema.
# Note: JSON uses lowercase null / false, not Python's None / False.
schema = """
{
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.fooddelivery.events.v1",
    "doc": "Represents a single state transition in the order lifecycle.",
    "fields": [
        {"name": "event_id",       "type": "string"},
        {"name": "order_id",       "type": "string"},
        {"name": "customer_id",    "type": "string"},
        {"name": "restaurant_id",  "type": "string"},
        {"name": "courier_id",     "type": ["null", "string"], "default": null},
        {"name": "zone_id",        "type": "string"},
        {
            "name": "event_type",
            "type": {
                "type": "enum",
                "name": "OrderEventType",
                "symbols": [
                    "ORDER_PLACED", "ORDER_ACCEPTED", "PREPARING",
                    "READY_FOR_PICKUP", "PICKED_UP", "DELIVERED",
                    "CANCELLED", "REFUND_ISSUED"
                ]
            }
        },
        {"name": "event_time",         "type": "string"},
        {"name": "ingestion_time",     "type": "string"},
        {"name": "order_value",        "type": "float"},
        {"name": "promo_code",         "type": ["null", "string"], "default": null},
        {"name": "promo_discount_pct", "type": ["null", "float"],  "default": null},
        {"name": "estimated_prep_min", "type": "int"},
        {"name": "is_late_event",      "type": "boolean", "default": false},
        {"name": "is_duplicate",       "type": "boolean", "default": false},
        {"name": "schema_version",     "type": "string",  "default": "1.0"}
    ]
}
"""


## 5. Kafka (Event Hub) Consumer Config

In [8]:
# Kafka/Event Hub consumer configuration
kafkaConf = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism":    "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config":  (
        f'org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" '
        f'password="{consumer_eventhub_connection_str}";'
    ),
    "subscribe":        eventhub_name,
    "startingOffsets":  "earliest",
    "groupIdPrefix":    "FoodDelivery_Orders_Analytics",
    "enable.auto.commit":       "true",
    "auto.commit.interval.ms":  "5000",
}

print("Kafka config ready for topic:", eventhub_name)


Kafka config ready for topic: group_10_orders


## 6. Read Stream + Decode Avro

In [9]:
# Read from Event Hub using the Kafka API
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafkaConf) \
    .load()

df.printSchema()


root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [10]:
# Deserialize the Avro payload stored in Kafka's 'value' column
df = df.select(from_avro(df.value, schema).alias("order"))
df.printSchema()


root
 |-- order: struct (nullable = true)
 |    |-- event_id: string (nullable = false)
 |    |-- order_id: string (nullable = false)
 |    |-- customer_id: string (nullable = false)
 |    |-- restaurant_id: string (nullable = false)
 |    |-- courier_id: string (nullable = true)
 |    |-- zone_id: string (nullable = false)
 |    |-- event_type: string (nullable = false)
 |    |-- event_time: string (nullable = false)
 |    |-- ingestion_time: string (nullable = false)
 |    |-- order_value: float (nullable = false)
 |    |-- promo_code: string (nullable = true)
 |    |-- promo_discount_pct: float (nullable = true)
 |    |-- estimated_prep_min: integer (nullable = false)
 |    |-- is_late_event: boolean (nullable = false)
 |    |-- is_duplicate: boolean (nullable = false)
 |    |-- schema_version: string (nullable = false)



In [11]:
from pyspark.sql.functions import col, to_timestamp

df = df.select(
    col("order.event_id"),
    col("order.order_id"),
    col("order.customer_id"),
    col("order.restaurant_id"),
    col("order.courier_id"),
    col("order.zone_id"),
    col("order.event_type"),
    col("order.event_time"),
    col("order.ingestion_time"),
    col("order.order_value"),
    col("order.promo_code"),
    col("order.promo_discount_pct"),
    col("order.estimated_prep_min"),
    col("order.is_late_event"),
    col("order.is_duplicate"),
    col("order.schema_version"),
)

df = df.withColumn("event_time_ts", to_timestamp(col("event_time")))

df.printSchema()


root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingestion_time: string (nullable = true)
 |-- order_value: float (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- promo_discount_pct: float (nullable = true)
 |-- estimated_prep_min: integer (nullable = true)
 |-- is_late_event: boolean (nullable = true)
 |-- is_duplicate: boolean (nullable = true)
 |-- schema_version: string (nullable = true)
 |-- event_time_ts: timestamp (nullable = true)



## 7. Smoke Test — verify events are flowing

In [12]:
!mkdir -p checkpoint output checkpoint/q1 checkpoint/q2 checkpoint/q3


In [13]:
# Smoke test: stream everything into an in-memory table so we can peek at it.
import time

query_name = "all_orders"

# If the query already exists from a previous run, skip re-creation
if query_name not in [q.name for q in spark.streams.active]:
    smoke_query = df.writeStream \
        .outputMode("append") \
        .format("memory") \
        .queryName(query_name) \
        .start()
    print(f"Streaming into in-memory table '{query_name}' — waiting 30s for events ...")
    time.sleep(30)
else:
    print(f"Stream '{query_name}' already running — just querying the table.")

print("\nSample events:")
spark.sql(f"SELECT * FROM {query_name} LIMIT 10").show(truncate=False)

print("\nTotal events received so far:")
spark.sql(f"SELECT count(*) AS n FROM {query_name}").show()


Streaming into in-memory table 'all_orders' — waiting 30s for events ...

Sample events:
+------------------------------------+------------------------------------+-----------+-------------+----------+----------+--------------+------------------------+------------------------+-----------+----------+------------------+------------------+-------------+------------+--------------+-----------------------+
|event_id                            |order_id                            |customer_id|restaurant_id|courier_id|zone_id   |event_type    |event_time              |ingestion_time          |order_value|promo_code|promo_discount_pct|estimated_prep_min|is_late_event|is_duplicate|schema_version|event_time_ts          |
+------------------------------------+------------------------------------+-----------+-------------+----------+----------+--------------+------------------------+------------------------+-----------+----------+------------------+------------------+-------------+------------+---

In [14]:
# List all active streaming queries
for q in spark.streams.active:
    print(f"Name: {q.name} | ID: {q.id} | Active: {q.isActive}")


Name: all_orders | ID: 6e9d8664-789c-4e31-a70d-050188ea1b44 | Active: True


## 8. Analytics Queries

Three use cases at increasing difficulty.
Each block below *starts* a streaming query and then *peeks* at its output
after a short wait. The queries all write to in-memory tables so you can
re-query them at any time with standard SQL:

```python
spark.sql("SELECT * FROM <table_name> WHERE ...").show()
```

Query helpers:


In [15]:
# Helper: wait N seconds and display the current state of an aggregate table.
# Use after starting a streaming query to peek at its output.

import time

def peek(query_name: str, order_by: str = "window_start DESC", n: int = 15, wait: int = 30):
    """Wait a bit for the query to accumulate output, then show top rows."""
    print(f"Waiting {wait}s for '{query_name}' to accumulate output ...")
    time.sleep(wait)
    try:
        spark.sql(f"SELECT * FROM {query_name} ORDER BY {order_by} LIMIT {n}").show(truncate=False)
        spark.sql(f"SELECT count(*) AS rows_in_table FROM {query_name}").show()
    except Exception as e:
        print(f"[WARN] Could not query table yet: {e}")


In [16]:
def stop_query(name: str) -> None:
    """Stop a single streaming query by name — useful when iterating on a query."""
    for q in spark.streams.active:
        if q.name == name:
            print(f"Stopping {name} ...")
            q.stop()
            return
    print(f"No active query named {name}.")


### 8.1 USE CASE 1 (BASIC) — Orders per zone — tumbling 5-min

In [17]:
# ─── USE CASE 1 (BASIC) ───────────────────────────────────────────────
# Orders placed per zone, in 5-minute tumbling windows.
#
# Tumbling windows = non-overlapping fixed intervals.
# Watermark = 10 minutes, so late events up to 10 min behind the latest
#             event_time are still counted in their correct window.

from pyspark.sql.functions import window, col, count, sum as _sum, when

orders_placed = df.filter(col("event_type") == "ORDER_PLACED") \
    .withWatermark("event_time_ts", "10 minutes") \
    .groupBy(
        window(col("event_time_ts"), "5 minutes"),
        col("zone_id"),
    ) \
    .agg(
        count("*").alias("orders_placed"),
        _sum("order_value").alias("revenue_eur"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("zone_id"),
        col("orders_placed"),
        col("revenue_eur"),
    )

q1_orders_per_zone = orders_placed.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("orders_per_zone_5min") \
    .option("checkpointLocation", "checkpoint/q1") \
    .start()

print("Started: orders_per_zone_5min")


Started: orders_per_zone_5min


In [18]:
peek("orders_per_zone_5min",
     order_by="window_start DESC, zone_id", n=20, wait=45)


Waiting 45s for 'orders_per_zone_5min' to accumulate output ...
+-------------------+-------------------+-----------+-------------+------------------+
|window_start       |window_end         |zone_id    |orders_placed|revenue_eur       |
+-------------------+-------------------+-----------+-------------+------------------+
|2026-04-18 18:05:00|2026-04-18 18:10:00|zone_north |1            |43.08000183105469 |
|2026-04-18 18:00:00|2026-04-18 18:05:00|zone_east  |2            |34.10000038146973 |
|2026-04-18 18:00:00|2026-04-18 18:05:00|zone_north |2            |61.19000053405762 |
|2026-04-18 17:55:00|2026-04-18 18:00:00|zone_north |1            |17.450000762939453|
|2026-04-18 17:55:00|2026-04-18 18:00:00|zone_west  |1            |26.1200008392334  |
|2026-04-18 17:50:00|2026-04-18 17:55:00|zone_centre|1            |35.849998474121094|
|2026-04-18 17:50:00|2026-04-18 17:55:00|zone_east  |3            |85.4399995803833  |
|2026-04-18 17:50:00|2026-04-18 17:55:00|zone_south |1            

### 8.2 USE CASE 2 (INTERMEDIATE) — Cancellation rate per zone — hopping 10-min / 5-min

In [19]:
# ─── USE CASE 2 (INTERMEDIATE) ────────────────────────────────────────
# Cancellation rate per zone in 10-minute hopping windows, stepping every 5 minutes.
# Hopping windows overlap — they give a smoother view of a rolling metric.
#
# We count total orders PLACED vs CANCELLED in each window.

from pyspark.sql.functions import window, col, count, when, round as _round

cancellations = df.filter(col("event_type").isin("ORDER_PLACED", "CANCELLED")) \
    .withWatermark("event_time_ts", "10 minutes") \
    .groupBy(
        window(col("event_time_ts"), "10 minutes", "5 minutes"),
        col("zone_id"),
    ) \
    .agg(
        count(when(col("event_type") == "ORDER_PLACED", True)).alias("orders_placed"),
        count(when(col("event_type") == "CANCELLED",    True)).alias("orders_cancelled"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("zone_id"),
        col("orders_placed"),
        col("orders_cancelled"),
        _round(
            when(col("orders_placed") > 0,
                 col("orders_cancelled") / col("orders_placed")
            ).otherwise(0.0),
            3
        ).alias("cancel_rate"),
    )

q2_cancel_rate = cancellations.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("cancel_rate_per_zone_10min") \
    .option("checkpointLocation", "checkpoint/q2") \
    .start()

print("Started: cancel_rate_per_zone_10min")


Started: cancel_rate_per_zone_10min


In [20]:
peek("cancel_rate_per_zone_10min",
     order_by="window_start DESC, zone_id", n=20, wait=45)


Waiting 45s for 'cancel_rate_per_zone_10min' to accumulate output ...
+-------------------+-------------------+-----------+-------------+----------------+-----------+
|window_start       |window_end         |zone_id    |orders_placed|orders_cancelled|cancel_rate|
+-------------------+-------------------+-----------+-------------+----------------+-----------+
|2026-04-18 18:10:00|2026-04-18 18:20:00|zone_east  |0            |1               |0.0        |
|2026-04-18 18:05:00|2026-04-18 18:15:00|zone_east  |0            |1               |0.0        |
|2026-04-18 18:05:00|2026-04-18 18:15:00|zone_north |1            |0               |0.0        |
|2026-04-18 18:00:00|2026-04-18 18:10:00|zone_east  |2            |0               |0.0        |
|2026-04-18 18:00:00|2026-04-18 18:10:00|zone_north |3            |0               |0.0        |
|2026-04-18 17:55:00|2026-04-18 18:05:00|zone_east  |2            |1               |0.5        |
|2026-04-18 17:55:00|2026-04-18 18:05:00|zone_north |3   

### 8.3 USE CASE 3 (ADVANCED) — Delivery time anomaly detection — 15-min windows

In [21]:
# ─── USE CASE 3 (ADVANCED) ────────────────────────────────────────────
# Delivery time anomaly detection per zone.
#
# Approach:
#   1. Pair each order's PICKED_UP and DELIVERED events via self-join on
#      order_id within a bounded time window (stream-stream join).
#   2. Compute delivery duration in minutes.
#   3. Aggregate mean/stddev of duration per zone in 15-minute tumbling
#      windows.
#   4. Flag windows where mean duration exceeds 45 min OR stddev > 20 min
#      (simple threshold heuristic — easier to reason about than rolling
#      z-scores over streaming aggregates).
#
# This demonstrates:
#   - stream-stream join with event-time constraint
#   - watermark propagation through multiple operators
#   - anomaly flagging logic

from pyspark.sql.functions import (
    col, window, expr, avg, stddev_pop, count, when, round as _round
)

pickups = df.filter(col("event_type") == "PICKED_UP") \
    .withWatermark("event_time_ts", "30 minutes") \
    .selectExpr(
        "order_id AS p_order_id",
        "zone_id  AS p_zone_id",
        "event_time_ts AS pickup_ts",
    )

deliveries = df.filter(col("event_type") == "DELIVERED") \
    .withWatermark("event_time_ts", "30 minutes") \
    .selectExpr(
        "order_id AS d_order_id",
        "event_time_ts AS delivery_ts",
    )

# Inner join on order_id, constrained so delivery must follow pickup by
# between 1 minute and 3 hours (discards impossible durations from the
# join, which we'll flag separately).
deliveries_enriched = pickups.join(
    deliveries,
    expr("""
        p_order_id = d_order_id AND
        delivery_ts >= pickup_ts AND
        delivery_ts <= pickup_ts + interval 3 hours
    """),
    "inner"
).withColumn(
    "delivery_minutes",
    (col("delivery_ts").cast("long") - col("pickup_ts").cast("long")) / 60.0
)

delivery_stats = deliveries_enriched \
    .groupBy(
        window(col("pickup_ts"), "15 minutes"),
        col("p_zone_id").alias("zone_id"),
    ) \
    .agg(
        count("*").alias("deliveries"),
        _round(avg("delivery_minutes"), 2).alias("avg_delivery_min"),
        _round(stddev_pop("delivery_minutes"), 2).alias("stddev_delivery_min"),
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("zone_id"),
        col("deliveries"),
        col("avg_delivery_min"),
        col("stddev_delivery_min"),
        when(
            (col("avg_delivery_min") > 45) | (col("stddev_delivery_min") > 20),
            True
        ).otherwise(False).alias("is_anomalous"),
    )

q3_delivery_anomalies = delivery_stats.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("delivery_anomalies_15min") \
    .option("checkpointLocation", "checkpoint/q3") \
    .start()

print("Started: delivery_anomalies_15min")


Started: delivery_anomalies_15min


In [22]:
peek("delivery_anomalies_15min",
     order_by="window_start DESC, zone_id", n=20, wait=60)

# Highlight only the anomalous windows
print("\nAnomalous windows only:")
spark.sql("""
    SELECT window_start, zone_id, deliveries,
           avg_delivery_min, stddev_delivery_min
    FROM delivery_anomalies_15min
    WHERE is_anomalous = true
    ORDER BY window_start DESC
    LIMIT 20
""").show(truncate=False)


Waiting 60s for 'delivery_anomalies_15min' to accumulate output ...
+-------------------+-------------------+------------+----------+----------------+-------------------+------------+
|window_start       |window_end         |zone_id     |deliveries|avg_delivery_min|stddev_delivery_min|is_anomalous|
+-------------------+-------------------+------------+----------+----------------+-------------------+------------+
|2026-04-18 18:30:00|2026-04-18 18:45:00|zone_north  |4         |29.0            |10.02              |false       |
|2026-04-18 18:30:00|2026-04-18 18:45:00|zone_south  |2         |18.0            |0.0                |false       |
|2026-04-18 18:15:00|2026-04-18 18:30:00|zone_centre |2         |30.0            |4.0                |false       |
|2026-04-18 18:15:00|2026-04-18 18:30:00|zone_east   |3         |25.67           |10.53              |false       |
|2026-04-18 18:15:00|2026-04-18 18:30:00|zone_south  |1         |9.0             |0.0                |false       |
|202

### 8.4 Exploration — run any SQL against the running tables

All four aggregate tables are now available as in-memory Spark tables.
Add cells here to explore the data interactively. Examples below.


In [ ]:
# Show all in-memory tables
spark.sql("SHOW TABLES").show(truncate=False)


In [ ]:
# Example: top revenue windows across all zones
# spark.sql('''
#     SELECT window_start, window_end, zone_id, revenue_eur
#     FROM orders_per_zone_5min
#     ORDER BY revenue_eur DESC
#     LIMIT 10
# ''').show(truncate=False)


## 9. Shutdown

In [ ]:
# Run this cell when you want to stop all streaming queries and the Spark session
if False:  # flip to True to actually stop
    for q in spark.streams.active:
        print(f"Stopping {q.name} ...")
        q.stop()
    spark.stop()
    print("Stopped.")
